# Aquire and Inspect data collected from Quantic Camera

In [ ]:
# Load QuantiCam library
using QuantiCam

## Acquire Frames from Live Camera

In [ ]:
# Setup connection to camera
if (@isdefined qc) && qc!== nothing
    # Try to reconfigure or cleanup and initialise from the begining
    try
        QuantiCam.reload_config(qc, "../config/tcspc.json")
        QuantiCam.config_sensor(qc)
    catch
        #QuantiCam.sensor_disconnect(qc)
        QuantiCam.cleanup!(qc)
    end
end
if !(@isdefined qc) || qc === nothing
    qc = load_qc(config_path="../config/tcspc.json")
end

In [ ]:
# Load new config if needed
new_config!(qc, "tcspc", "../config/tcspc.json")

In [ ]:
# Initialise the HDF5 (optional)
hdf5_filename = "../experiment/board_calibration.h5"
hdf5_task, hdf5_channel = QuantiCam.hdf5_collector_init(hdf5_filename, Matrix{UInt16}; description="Board placed at different distances for calibration")

In [ ]:
# Create dataset group
number_of_frames = 1000
group_config = QuantiCam.GroupConfig("7", number_of_frames, "board placed at 45cm")
put!(hdf5_channel, group_config)

In [ ]:
# Acqire TCSPC histograms from live camera
frames = QuantiCam.capture_frames(qc, number_of_frames; hdf_channel=hdf5_channel)
filtered_frames = map(frame -> QuantiCam.filter_code(frame, 0xffc), frames)
typeof(filtered_frames)

In [ ]:
# Clean up
QuantiCam.cleanup!(qc)

## Load Data from HDF5 Files

In [ ]:
# Alternatively, load from HDF5 file
using HDF5

hdf5_filename = "../experiment/board_calibration.h5"
fid = h5open(hdf5_filename, "r")
group = fid["1"]
frames_h5 = read_dataset(group, "frames")
frames = collect(Matrix{UInt16}, eachslice(frames_h5, dims=1));
close(fid)
filtered_frames = map(frame -> QuantiCam.filter_code(frame, 0xffc), frames)
typeof(filtered_frames)

In [ ]:
compensation_filename = "../experiment/compensated_depth_map.h5"
compensation_frame = h5read(compensation_filename, "compensation");

## Process and Visualise the Data

In [ ]:
using Statistics
using Plots

# Collect frames into per-pixel vectors
tcspc_stream = QuantiCam.collect_frames(filtered_frames)
tcspc_mean = map(pixel -> mean(pixel), tcspc_stream)
tcspc_var = map(pixel -> var(pixel), tcspc_stream)

# Visualise the number of returns for each pixel
tcspc_events = map(pixel -> length(collect((skipmissing(pixel)))), tcspc_stream)
heatmap(tcspc_events[:,10:118], aspect_ratio = 0.5)

In [ ]:
# Visualise a single pixel histogram
histogram(tcspc_stream[81, 64], bins=200)

In [ ]:
# Visualise the histogram of all pixel values
all_pixels = collect(Iterators.filter(x-> x>2.0, skipmissing(Iterators.flatten(tcspc_stream))))
histogram(all_pixels, bins=200)

In [ ]:
using Plots

time_histograms = QuantiCam.build_histogram(tcspc_stream, 256)
#time_histograms_cropped = [time_histograms[i, j][(1):(256)] for i in 1:size(time_histograms, 1), j in 1:size(time_histograms, 2)]
time_histograms_cropped = copy(time_histograms)
for i in 1:size(time_histograms, 1), j in 1:size(time_histograms, 2)
    time_histograms_cropped[i, j][1:10] .= 0
end
centroids = QuantiCam.decode_histogram_to_depth(time_histograms_cropped, 10)
#centroids = QuantiCam.decode_histogram_to_depth(time_histograms_cropped, 10, compensation_frame)
min_val, min_idx = findmin(ifelse.(isnan.(centroids), Inf, centroids)[:,1:128])
max_val, max_idx = findmax(ifelse.(isnan.(centroids), -Inf, centroids)[:,1:128])
avg_centroid_value = mean(filter(!isnan, centroids))
median_centroid_value = median(filter(!isnan, centroids))
println("Min depth: $min_val at index $min_idx")
println("Max depth: $max_val at index $max_idx")
println("Average depth: $avg_centroid_value")
println("Median depth: $median_centroid_value")
println("Depth range: $(max_val - min_val)")
intensity_img = heatmap(tcspc_events[:,2:127], colorbar=true, aspect_ratio = 0.5, title="Intensity Map")
depth_img = heatmap(centroids[:,2:127], colorbar=true, aspect_ratio=0.5, title="Depth Map")
plot(intensity_img, depth_img, layout = (1, 2), size=(1000, 400))

In [ ]:
# Visualise (Sanity check)
i, j = 189, 105
h = time_histograms[i, j]
print(centroids[i, j])
plot(1:length(h), h,
     xlabel="Bin Index",
     ylabel="Photon Count",
     title="Histogram at pixel ($i,$j)",
     legend=false)

In [ ]:
# Visualise the distribution of depth values
histogram(vec(centroid), bins=1000,
          xlabel="Depth Value",
          ylabel="Number of Pixels",
          title="Distribution of Depth Values",
          legend=false)

In [ ]:
# Plot the column against the mean column value of centroid
avg_column = mean(centroid[:,2:127], dims=1)
#print the shape of avg_column
size(avg_column)
plot(1:length(avg_column), avg_column[:],
     xlabel="Column Index",
     ylabel="Mean Depth Value",
     title="Mean Depth Value Across Columns",
     legend=false)

In [ ]:
# Shift all pixels so that centroids value are the same
avg_centoid_value = mean(filter(!isnan, centroids))
print(avg_centoid_value)
compensation_frame = avg_centoid_value .- centroids
centroids_compensated = centroids .+ compensation_frame
heatmap(compensation_frame, colorbar=true, aspect_ratio=0.5, title="Compensation Frame")

In [ ]:
# Save compensation frame
if isfile("../experiment/compensated_depth_map.h5")
    println("File already exists!")
else
    h5open("../experiment/compensated_depth_map.h5", "w") do fid
        write(fid, "compensation", compensation_frame)
    end
end
heatmap(compensation_frame[:,2:127], colorbar=true, aspect_ratio=0.5, title="Compensation Frame")